In [1]:
import pytesseract

pytesseract.pytesseract.tesseract_cmd = (
    r"C:\Program Files\Tesseract-OCR\tesseract.exe"
)


In [2]:
from pdf2image import convert_from_path
from PIL import Image
import os

def load_images(file_path):
    images = []
    
    if file_path.lower().endswith(".pdf"):
        pages = convert_from_path(file_path, dpi=300)
        images.extend(pages)
    else:
        images.append(Image.open(file_path))
    
    return images


In [3]:
import cv2
import numpy as np

def preprocess_image(pil_image):
    img = np.array(pil_image)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # Noise removal
    blur = cv2.GaussianBlur(gray, (5,5), 0)
    
    # Adaptive thresholding
    thresh = cv2.adaptiveThreshold(
        blur, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY, 11, 2
    )
    
    return thresh


In [4]:
def extract_text(processed_img):
    config = "--oem 3 --psm 6"
    text = pytesseract.image_to_string(
        processed_img,
        config=config
    )
    return text


In [5]:
import re

def clean_text(text):
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text)
    return text


In [6]:
def extract_medical_values(text):
    data = {}

    patterns = {
        "Age": r"Age[:\s]+(\d{1,3})",
        "RestingBP": r"(BP|Blood Pressure)[:\s]+(\d{2,3})",
        "Cholesterol": r"Cholesterol[:\s]+(\d{2,4})",
        "FastingBS": r"(FBS|Fasting Blood Sugar)[:\s]+(\d)"
    }

    for key, pattern in patterns.items():
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            data[key] = int(match.groups()[-1])

    return data


In [7]:
def ocr_pipeline(file_path):
    images = load_images(file_path)
    extracted_data = {}

    for img in images:
        processed = preprocess_image(img)
        text = extract_text(processed)
        clean = clean_text(text)
        data = extract_medical_values(clean)
        extracted_data.update(data)

    return extracted_data


In [15]:
output = ocr_pipeline("sample_report.png")
print(output)


UnidentifiedImageError: cannot identify image file 'sample_report.png'